In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import torch
from torch.utils.data import Dataset, DataLoader
# Fix for AdamW import
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Create folders for saving models and graphs
os.makedirs('saved_models', exist_ok=True)
os.makedirs('graphs', exist_ok=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load the data
train_df = pd.read_csv('training.csv')
val_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

# Data exploration and visualization
def visualize_data(train_df, val_df, test_df):
    # Count labels distribution
    plt.figure(figsize=(12, 5))

    # Train data
    plt.subplot(131)
    sns.countplot(x='label', data=train_df)
    plt.title('Label Distribution - Training Data')

    # Validation data
    plt.subplot(132)
    sns.countplot(x='label', data=val_df)
    plt.title('Label Distribution - Validation Data')

    # Test data
    plt.subplot(133)
    sns.countplot(x='label', data=test_df)
    plt.title('Label Distribution - Test Data')

    plt.tight_layout()
    plt.savefig('graphs/label_distribution.png')
    plt.close()

    # Text length distribution
    plt.figure(figsize=(12, 5))

    # Train data
    plt.subplot(131)
    train_df['text_length'] = train_df['text'].apply(lambda x: len(str(x).split()))
    sns.histplot(data=train_df, x='text_length', bins=50)
    plt.title('Text Length - Training Data')
    plt.xlabel('Number of Words')

    # Validation data
    plt.subplot(132)
    val_df['text_length'] = val_df['text'].apply(lambda x: len(str(x).split()))
    sns.histplot(data=val_df, x='text_length', bins=50)
    plt.title('Text Length - Validation Data')
    plt.xlabel('Number of Words')

    # Test data
    plt.subplot(133)
    test_df['text_length'] = test_df['text'].apply(lambda x: len(str(x).split()))
    sns.histplot(data=test_df, x='text_length', bins=50)
    plt.title('Text Length - Test Data')
    plt.xlabel('Number of Words')

    plt.tight_layout()
    plt.savefig('graphs/text_length_distribution.png')
    plt.close()

    # Print dataset statistics
    print(f"Training data shape: {train_df.shape}")
    print(f"Validation data shape: {val_df.shape}")
    print(f"Test data shape: {test_df.shape}")

    # Count number of classes
    num_classes = len(train_df['label'].unique())
    print(f"Number of classes: {num_classes}")
    print(f"Classes: {sorted(train_df['label'].unique())}")

# Run visualization
visualize_data(train_df, val_df, test_df)

# Get the number of classes
num_classes = len(train_df['label'].unique())

# Create a custom dataset
class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = dataframe['text'].values
        self.labels = dataframe['label'].values

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Initialize the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = EmotionDataset(train_df, tokenizer)
val_dataset = EmotionDataset(val_df, tokenizer)
test_dataset = EmotionDataset(test_df, tokenizer)

# Create data loaders
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

# Initialize the BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_classes,
    output_attentions=False,
    output_hidden_states=False
)

# Move model to device
model.to(device)

# Set up optimizer and scheduler
# Changed from transformers.AdamW to torch.optim.AdamW
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

# Number of training epochs
epochs = 10

# Total number of training steps
total_steps = len(train_loader) * epochs

# Set up the learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Training loop
def train_model():
    best_val_acc = 0.0
    training_stats = []

    for epoch in range(epochs):
        print(f'\nEpoch {epoch+1}/{epochs}')
        print('-' * 40)

        # Training phase
        model.train()
        total_train_loss = 0
        train_correct = 0
        train_total = 0

        # Create progress bar
        train_progress_bar = tqdm(train_loader, desc="Training")

        for batch in train_progress_bar:
            # Clear gradients
            optimizer.zero_grad()

            # Get inputs
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            # Calculate accuracy
            _, predicted = torch.max(logits, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            # Backward pass
            loss.backward()

            # Clip gradients to avoid exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Update weights
            optimizer.step()

            # Update learning rate
            scheduler.step()

            # Add batch loss to total loss
            total_train_loss += loss.item()

            # Update progress bar
            train_progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        # Calculate average training loss and accuracy
        avg_train_loss = total_train_loss / len(train_loader)
        train_accuracy = train_correct / train_total
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Train Accuracy: {train_accuracy:.4f}")

        # Validation phase
        model.eval()
        total_val_loss = 0
        val_preds = []
        val_true = []

        # Create progress bar for validation
        val_progress_bar = tqdm(val_loader, desc="Validation")

        with torch.no_grad():
            for batch in val_progress_bar:
                # Get inputs
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                # Forward pass
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss
                logits = outputs.logits

                # Add batch loss to total loss
                total_val_loss += loss.item()

                # Convert logits to predictions
                _, preds = torch.max(logits, 1)

                # Append predictions and true labels
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.cpu().numpy())

                # Update progress bar
                val_progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        # Calculate average validation loss and accuracy
        avg_val_loss = total_val_loss / len(val_loader)
        val_accuracy = accuracy_score(val_true, val_preds)
        print(f"Validation Loss: {avg_val_loss:.4f}")
        print(f"Validation Accuracy: {val_accuracy:.4f}")

        # Save model after each epoch
        torch.save(model.state_dict(), f'saved_models/model_epoch_{epoch+1}.pt')

        # Save the best model
        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            torch.save(model.state_dict(), 'saved_models/best_model.pt')
            print(f"Best model saved with accuracy: {best_val_acc:.4f}")

        # Store stats
        training_stats.append({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'train_acc': train_accuracy,
            'val_loss': avg_val_loss,
            'val_acc': val_accuracy
        })

    # Plot training and validation loss
    epochs_range = list(range(1, epochs + 1))
    train_loss_values = [stat['train_loss'] for stat in training_stats]
    val_loss_values = [stat['val_loss'] for stat in training_stats]

    plt.figure(figsize=(10, 6))
    plt.plot(epochs_range, train_loss_values, label='Training Loss')
    plt.plot(epochs_range, val_loss_values, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('graphs/loss_curves.png')
    plt.close()

    # Plot training and validation accuracy
    train_acc_values = [stat['train_acc'] for stat in training_stats]
    val_acc_values = [stat['val_acc'] for stat in training_stats]

    plt.figure(figsize=(10, 6))
    plt.plot(epochs_range, train_acc_values, label='Training Accuracy')
    plt.plot(epochs_range, val_acc_values, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig('graphs/accuracy_curves.png')
    plt.close()

    return training_stats

# Train the model
training_stats = train_model()

# Evaluate on test data
def evaluate_model():
    print("\nEvaluating model on test data...")

    # Load the best model
    model.load_state_dict(torch.load('saved_models/best_model.pt'))
    model.eval()

    test_preds = []
    test_true = []

    # Create progress bar for testing
    test_progress_bar = tqdm(test_loader, desc="Testing")

    with torch.no_grad():
        for batch in test_progress_bar:
            # Get inputs
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            # Convert logits to predictions
            _, preds = torch.max(logits, 1)

            # Append predictions and true labels
            test_preds.extend(preds.cpu().numpy())
            test_true.extend(labels.cpu().numpy())

    # Calculate accuracy
    test_accuracy = accuracy_score(test_true, test_preds)
    print(f"Test Accuracy: {test_accuracy:.4f}")

    # Print classification report
    print("\nClassification Report:")
    print(classification_report(test_true, test_preds))

    # Create confusion matrix
    cm = confusion_matrix(test_true, test_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.savefig('graphs/confusion_matrix.png')
    plt.close()

    return test_accuracy, test_true, test_preds

# Evaluate the model
test_accuracy, test_true, test_preds = evaluate_model()

print("\nTask completed successfully!")
print(f"Best model saved at: saved_models/best_model.pt")
print(f"All models saved in the 'saved_models' directory")
print(f"All graphs saved in the 'graphs' directory")

Using device: cuda
Training data shape: (16000, 3)
Validation data shape: (2000, 3)
Test data shape: (2000, 3)
Number of classes: 6
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Train Loss: 0.0105
Train Accuracy: 0.9969


Validation: 100%|██████████| 125/125 [00:14<00:00,  8.44it/s, loss=0.1327]


Validation Loss: 0.3687
Validation Accuracy: 0.9400

Evaluating model on test data...


Testing: 100%|██████████| 125/125 [00:14<00:00,  8.50it/s]


Test Accuracy: 0.9295

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       581
           1       0.94      0.96      0.95       695
           2       0.89      0.79      0.84       159
           3       0.94      0.91      0.93       275
           4       0.92      0.85      0.88       224
           5       0.70      0.79      0.74        66

    accuracy                           0.93      2000
   macro avg       0.89      0.88      0.88      2000
weighted avg       0.93      0.93      0.93      2000


Task completed successfully!
Best model saved at: saved_models/best_model.pt
All models saved in the 'saved_models' directory
All graphs saved in the 'graphs' directory


In [4]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import numpy as np
import os

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Class to label mapping - Updated to include 6 classes
# Note: You may need to adjust this mapping based on your actual training data
class_mapping = {
    0: "Sadness",
    1: "Joy",
    2: "Love",
    3: "Anger",
    4: "Fear",
    5: "Other"  # Added the 6th class - adjust the name as needed
}

def load_model(model_path):
    """Load the trained BERT model with the correct number of labels"""
    # Get the number of labels from the saved model
    try:
        # First try loading the model to inspect its structure
        temp_state_dict = torch.load(model_path, map_location=device)
        # Check the classifier.bias size to determine number of classes
        num_labels = temp_state_dict['classifier.bias'].size(0)
        print(f"Detected {num_labels} classes in the saved model")
    except Exception as e:
        print(f"Could not automatically detect number of classes: {e}")
        print("Defaulting to 6 classes based on the error message")
        num_labels = 6

    # Initialize the model with the correct architecture
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=num_labels,
        output_attentions=False,
        output_hidden_states=False
    )

    # Load saved weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    return model, num_labels

def predict_emotion(text, model, tokenizer, max_len=128):
    """Predict emotion for a single text input"""
    # Preprocess the text - same preprocessing as during training
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    # Move tensors to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Get prediction
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    # Convert logits to probabilities
    probabilities = torch.nn.functional.softmax(logits, dim=1)

    # Get predicted class
    pred_class = torch.argmax(probabilities, dim=1).cpu().numpy()[0]

    # Get class probabilities
    probs = probabilities.cpu().numpy()[0]

    # Create a dictionary of probabilities for all classes
    prob_dict = {}
    for i in range(len(probs)):
        class_name = class_mapping.get(i, f"Class {i}")
        prob_dict[class_name] = float(probs[i])

    return {
        'predicted_class': int(pred_class),
        'predicted_emotion': class_mapping.get(pred_class, f"Class {pred_class}"),
        'probabilities': prob_dict
    }

def main():
    # Path to your saved model
    model_path = input("Enter the path to your saved model (default: saved_models/best_model.pt): ")
    if not model_path:
        model_path = 'saved_models/best_model.pt'

    if not os.path.exists(model_path):
        print(f"Error: Model file '{model_path}' not found.")
        return

    try:
        # Initialize tokenizer - same as used during training
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

        # Load model with correct number of classes
        print("\nLoading model...")
        model, num_labels = load_model(model_path)
        print("Model loaded successfully!")

        # Update class mapping if needed
        if num_labels != len(class_mapping):
            print(f"Warning: Model has {num_labels} classes but mapping only has {len(class_mapping)} classes.")
            print("Some predictions may show as 'Class X' instead of emotion names.")

        print("\nEmotion Prediction System")
        print(f"Emotions: {', '.join([f'{v} ({k})' for k, v in class_mapping.items()])}")

        # Interactive prediction loop
        while True:
            # Get user input
            user_text = input("\nEnter text to predict emotion (or 'q' to quit): ")

            if user_text.lower() == 'q':
                break

            if not user_text.strip():
                print("Please enter some text.")
                continue

            # Predict emotion
            result = predict_emotion(user_text, model, tokenizer)

            # Display results
            print(f"\nPredicted Emotion: {result['predicted_emotion']}")
            print("\nProbabilities:")
            for emotion, prob in result['probabilities'].items():
                print(f"  {emotion}: {prob:.4f}")

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Using device: cuda
Enter the path to your saved model (default: saved_models/best_model.pt): 

Loading model...
Detected 6 classes in the saved model


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!

Emotion Prediction System
Emotions: Sadness (0), Joy (1), Love (2), Anger (3), Fear (4), Other (5)

Enter text to predict emotion (or 'q' to quit): i felt anger when at the end of a telephone call

Predicted Emotion: Anger

Probabilities:
  Sadness: 0.0020
  Joy: 0.0009
  Love: 0.0005
  Anger: 0.9911
  Fear: 0.0054
  Other: 0.0002

Enter text to predict emotion (or 'q' to quit): q


In [6]:
import zipfile
import os

def zip_folders(folders, zip_filename):
    """Zips multiple folders into a single zip file.

    Args:
        folders: A list of folder paths to include in the zip file.
        zip_filename: The name of the zip file to create.
    """
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for folder in folders:
            # Iterate over all files in each folder
            for root, _, files in os.walk(folder):
                for file in files:
                    # Create the relative path for the file within the zip
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, os.path.commonprefix([folder, file_path]))
                    zipf.write(file_path, arcname=arcname)

# Example usage:
folders_to_zip = ['saved_models', 'graphs']
zip_filename = 'outputs.zip'

zip_folders(folders_to_zip, zip_filename)
print(f"Folders '{folders_to_zip[0]}' and '{folders_to_zip[1]}' zipped to '{zip_filename}'")

Folders 'saved_models' and 'graphs' zipped to 'outputs.zip'
